# Phase 2 comparison -- filtered vs. unfiltered Harbin models

**Run this only after both `code_phase2_harbin_unfiltered.ipynb` and
`code_phase2_harbin_filtered.ipynb` have finished training and saved
their models to Drive.** This notebook is fully self-contained (its own
Drive mount, its own extraction of both datasets, its own model loading)
-- it doesn't depend on either training notebook's session state, so it's
safe to run in a completely fresh Colab runtime.

**What's being compared and why it's a fair comparison**: both datasets
share the identical 917/211/184 train/val/test split (T52TDT excluded
from both, split assignment doesn't depend on the filter) -- so this
pairs up predictions on the exact same validation images across both
models. Each model is evaluated against its *own* dataset's labels
(filtered model vs. filtered masks, unfiltered model vs. unfiltered
masks) -- since neither is externally verified ground truth (see
`poster_notes.md`'s limitations discussion), this tests how well each
model learned to reproduce its respective labelling scheme, not which
labelling scheme is objectively "true." A paired Wilcoxon signed-rank
test (non-parametric, appropriate since per-image accuracy isn't
guaranteed to be normally distributed) checks whether the difference in
per-image accuracy between the two models is statistically significant,
addressing the coursework's "conduct statistical significance testing"
requirement (Part A.5c).

In [ ]:
import os
# Force legacy Keras 2 behaviour. Current TF ships Keras 3 by default, which
# breaks this codebase's private optimizer-internals usage and the
# unmaintained `segmentation_models` package -- tf_keras is TF's official
# compatibility shim for exactly this situation, same fix already applied
# in predictor.py.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import sys
import numpy as np
import pandas as pd
import PIL
import requests
import tensorflow as tf
import tf_keras as keras
from tf_keras.models import *
from tf_keras.layers import *
from tf_keras.optimizers import *
from tf_keras.losses import *
from tf_keras import backend as K
from tf_keras.callbacks import ModelCheckpoint
from tf_keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import *


In [ ]:
# Colab setup: mount Drive, clone the repo, extract BOTH ground-truth
# archives, and locate both trained models (copied to Drive at the end
# of each training notebook).
import os
import shutil

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

MODEL_FILENAMES = {
    "unfiltered": "unet-attention-4d-harbin-unfiltered.hdf5",
    "filtered": "unet-attention-4d-harbin-filtered.hdf5",
}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = "/content/COMP0173_poster_pre"
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/jy-gfm/COMP0173_poster_pre.git {REPO_DIR}
    os.chdir(REPO_DIR)

    DRIVE_DIR = "/content/drive/MyDrive/COMP0173/"
    DATA_DIRS = {}
    MODEL_PATHS = {}

    for version in ("unfiltered", "filtered"):
        tar_name = f"Haerbing_ground_truth_{version}"
        local_data_dir = f"/content/{tar_name}"
        if not os.path.exists(local_data_dir):
            extract_root = f"/content/{tar_name}_extract"
            os.makedirs(extract_root, exist_ok=True)
            !tar -xf {DRIVE_DIR}{tar_name}.tar -C {extract_root} 2>&1 | grep -v "Ignoring unknown extended header keyword" || true
            local_data_dir = f"{extract_root}/{tar_name}"
        DATA_DIRS[version] = local_data_dir

        local_model_path = f"/content/{MODEL_FILENAMES[version]}"
        if not os.path.exists(local_model_path):
            shutil.copy(f"{DRIVE_DIR}{MODEL_FILENAMES[version]}", local_model_path)
        MODEL_PATHS[version] = local_model_path
else:
    DATA_DIRS = {v: f"./Haerbing_ground_truth_{v}" for v in ("unfiltered", "filtered")}
    MODEL_PATHS = {v: MODEL_FILENAMES[v] for v in ("unfiltered", "filtered")}

print("DATA_DIRS:", DATA_DIRS)
print("MODEL_PATHS:", MODEL_PATHS)


## Load both models and run the paired comparison

In [ ]:
import numpy as np
import glob
from scipy.stats import wilcoxon

def load_validation(version):
    paths = sorted(glob.glob(f"{DATA_DIRS[version]}/validation/images/*.npy"))
    images = [np.load(p).reshape(1, 512, 512, 4) for p in paths]
    masks = [np.load(p.replace("/images/", "/masks/")).reshape(1, 512, 512, 1) for p in paths]
    return paths, images, masks

def per_image_accuracy(model, images, masks):
    scores = []
    for img, mask in zip(images, masks):
        pred = np.round(model.predict(img, verbose=0)).flatten()
        scores.append((pred == mask.flatten()).mean())
    return np.array(scores)

model_unfiltered = keras.models.load_model(MODEL_PATHS["unfiltered"], compile=False)
model_filtered = keras.models.load_model(MODEL_PATHS["filtered"], compile=False)

paths_u, images_u, masks_u = load_validation("unfiltered")
paths_f, images_f, masks_f = load_validation("filtered")

# Sanity check: same underlying positions in the same order in both datasets
assert [os.path.basename(p) for p in paths_u] == [os.path.basename(p) for p in paths_f], \
    "validation sets don't line up between the two versions -- can't pair them"

acc_unfiltered = per_image_accuracy(model_unfiltered, images_u, masks_u)
acc_filtered = per_image_accuracy(model_filtered, images_f, masks_f)

print(f"unfiltered model: mean per-image accuracy = {acc_unfiltered.mean():.4f}")
print(f"filtered model:   mean per-image accuracy = {acc_filtered.mean():.4f}")

stat, p_value = wilcoxon(acc_unfiltered, acc_filtered)
print(f"\nWilcoxon signed-rank test: statistic={stat:.2f}, p-value={p_value:.4g}")
if p_value < 0.05:
    print("Difference is statistically significant at alpha=0.05.")
else:
    print("No statistically significant difference at alpha=0.05.")


## Visualize the comparison

Three complementary views: mean accuracy with spread, the full
per-image distribution, and a paired scatter (each point is one
validation image, comparing that same image's accuracy under both
models) -- the scatter directly visualizes what the Wilcoxon test above
is testing: points above the diagonal are images where the filtered
model did better, points below are where unfiltered did better.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Mean accuracy with std error bars
means = [acc_unfiltered.mean(), acc_filtered.mean()]
stds = [acc_unfiltered.std(), acc_filtered.std()]
axes[0].bar(["unfiltered", "filtered"], means, yerr=stds, capsize=8, color=["#8899AA", "#3B7A57"])
axes[0].set_ylabel("mean per-image accuracy")
axes[0].set_title("Mean accuracy (error bars = std)")
axes[0].set_ylim(0, 1)

# 2. Full distribution of per-image accuracy
axes[1].boxplot([acc_unfiltered, acc_filtered], labels=["unfiltered", "filtered"])
axes[1].set_ylabel("per-image accuracy")
axes[1].set_title("Distribution of per-image accuracy")

# 3. Paired per-image scatter (same validation image in both models)
lims = [0, 1]
axes[2].plot(lims, lims, 'k--', linewidth=1, label="y = x")
axes[2].scatter(acc_unfiltered, acc_filtered, alpha=0.5, s=15)
axes[2].set_xlabel("unfiltered model accuracy")
axes[2].set_ylabel("filtered model accuracy")
axes[2].set_title(f"Paired per-image comparison\n(Wilcoxon p={p_value:.4g})")
axes[2].legend()
axes[2].set_xlim(lims)
axes[2].set_ylim(lims)

plt.tight_layout()
plt.show()

print(f"unfiltered: {acc_unfiltered.mean():.4f} +/- {acc_unfiltered.std():.4f}")
print(f"filtered:   {acc_filtered.mean():.4f} +/- {acc_filtered.std():.4f}")
n_filtered_better = int((acc_filtered > acc_unfiltered).sum())
n_unfiltered_better = int((acc_unfiltered > acc_filtered).sum())
n_tied = len(acc_unfiltered) - n_filtered_better - n_unfiltered_better
print(f"filtered better on {n_filtered_better}/{len(acc_unfiltered)} images, "
      f"unfiltered better on {n_unfiltered_better}, tied on {n_tied}")
